# 레이크하우스 접속 스타터

호스트 Jupyter에서 kind 클러스터의 Iceberg 레이크하우스에 붙는 최소 예제.
**계산이 어느 파드에서 도는지**를 관측하는 예제는 [`01-spark-on-k8s.ipynb`](01-spark-on-k8s.ipynb)에 있다.

## 전제 ① — 서버를 먼저 올린다

Spark Connect는 **평시 `--replicas=0`** 이다(상주 컴퓨트라 회수 규율이 걸려 있다).
붙기 전에 올리고, 다 쓰면 내린다.

```shell
kubectl scale deploy/spark-connect --replicas=1
kubectl get pods -l spark-role=executor   # `k8s://` 이후로는 executor도 함께 뜬다
```

## 전제 ② — 접속 경로는 TLS Ingress가 기본, port-forward는 폴백

gRPC를 **TLS Ingress**로 내는 것이 규약이고(`docs/conventions/k8s.md` §10),
`port-forward`는 CA 미배포 환경·컨트롤러 장애용 **폴백**이다.
⚠️ 데이터 경로를 Ingress에 묶으면 컨트롤러 가용성에 종속된다 — port-forward에는 없던 결합이다.

```shell
# 경로 ① TLS Ingress (기본) — .env에 두 값이 함께 있어야 한다
#   SPARK_REMOTE=sc://spark-grpc.localtest.me:8443/;use_ssl=true
#   GRPC_DEFAULT_SSL_ROOTS_FILE_PATH=~/.lakehouse-ca.crt
kubectl get secret spark-grpc-tls -o jsonpath='{.data.ca\.crt}' | base64 -d > ~/.lakehouse-ca.crt

# 경로 ② port-forward (폴백)
kubectl port-forward svc/spark-connect 15002:15002       # Spark SQL
kubectl port-forward svc/catalog-postgres-rw 15432:5432  # 선택 (pyiceberg 직접 접속)
kubectl port-forward svc/seaweedfs 18333:8333            # 선택 (pyiceberg 직접 접속)
```

🔴 `sc://` URL에는 **CA를 지정하는 옵션이 없다** — 신뢰 주입은 `GRPC_DEFAULT_SSL_ROOTS_FILE_PATH`
환경변수로만 된다.

카탈로그 Postgres는 **CloudNativePG**가 관리하므로 서비스명에 `-rw`(쓰기)·`-ro`(읽기 전용)
접미사가 붙는다. 접미사 없는 `catalog-postgres`는 **존재하지 않는다**(`docs/conventions/k8s.md` §12).

## Trino는 쓰지 않는다

재설계에서 Trino는 제거 대상이고 ad-hoc 조회는 **Spark SQL**로 간다
(`docs/architectures/trino.md`). compose의 `trino`는 `--profile legacy-sql` 로만 뜬다.

## ⚠️ 보안 — 셀 출력

원천은 **비식별 연구 데이터셋이지만 DUA 대상**이다(`docs/security.md`).
셀 출력에 남은 행이 그대로 커밋되지 않도록 `nbstripout` pre-commit 훅이 걸려 있다.
훅을 우회해 강제 커밋하지 않는다.

## 1. 환경 로드와 사전 점검

🔴 `.env`는 **맨 앞에서** 읽는다. 아래쪽에서 읽으면 그 위의 셀은 `.env`를 못 본 채 기본값으로
접속하고, **그래도 돌기 때문에** 틀린 대상에 붙은 것을 눈치채지 못한다.

사전 점검은 `SPARK_REMOTE`를 **파싱해서** 그 대상을 본다 — 포트를 하드코딩하면
TLS Ingress 경로일 때 **정상인데 `MISSING`** 이 뜬다(거짓 음성).

In [ ]:
import os
import socket
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

# repo 루트의 .env를 읽어 os.environ에 주입한다(노트북 cwd = notebooks/).
# 이미 셸에서 export한 값은 덮지 않는다(load_dotenv 기본 override=False).
ENV_PATH = Path.cwd().parent / ".env"
loaded = load_dotenv(ENV_PATH)
SPARK_REMOTE = os.environ.get("SPARK_REMOTE", "sc://localhost:15002")

# `sc://host:port/;use_ssl=true` 형태라 `;` 로 먼저 자른 뒤 파싱한다
# (그대로 urlparse에 넣으면 포트에 `/;use_ssl=true`가 붙어 파싱이 깨진다).
parsed = urlparse(SPARK_REMOTE.split(";")[0].rstrip("/"))
spark_host = parsed.hostname or "localhost"
spark_port = parsed.port or 15002
via_ingress = spark_host not in ("localhost", "127.0.0.1")

print(f".env      : {ENV_PATH} ({'읽음' if loaded else '없음 — 기본값으로 간다'})")
print(f"접속 대상 : {SPARK_REMOTE}")
print(f"경로      : {'TLS Ingress' if via_ingress else 'port-forward(폴백)'}")
if via_ingress and not os.environ.get("GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"):
    print("🔴 GRPC_DEFAULT_SSL_ROOTS_FILE_PATH 미설정 — 자체서명 CA 검증에 실패한다")

# 대상: (레이블, 호스트, 포트, 필수 여부). Spark는 위에서 정한 경로를 그대로 본다.
# 카탈로그 PG는 CloudNativePG가 만드는 `catalog-postgres-rw`로 port-forward한다.
ENDPOINTS = [
    ("spark-connect", spark_host, spark_port, True),
    ("catalog-postgres-rw", "127.0.0.1", 15432, False),
    ("seaweedfs(s3)", "127.0.0.1", 18333, False),
]

for label, host, port, required in ENDPOINTS:
    with socket.socket() as sock:
        sock.settimeout(2)
        alive = sock.connect_ex((host, port)) == 0
    mark = "OK" if alive else ("MISSING" if required else "skip")
    print(f"[{mark:>7}] {label:<21} {host}:{port}")

## 2. Spark Connect 접속

카탈로그 설정(JDBC URI·warehouse·S3 엔드포인트·자격증명)은 **서버 측**
`k8s/spark/spark-connect-server.yaml`에 이미 들어 있다.
→ 클라이언트는 주소만 알면 되고 **비밀정보를 노트북에 두지 않는다**.

In [ ]:
from pyspark.sql import SparkSession

# SPARK_REMOTE는 §1에서 .env를 읽어 이미 정해 두었다.
spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
print("Spark", spark.version)
print("master:", spark.conf.get("spark.master", "<못 읽음>"))

## 3. 카탈로그 탐색

카탈로그 이름은 **전 엔진 `iceberg`로 통일**한다. JDBC 카탈로그는 `catalog_name`으로
레지스트리를 분할하므로, 이름이 다르면 같은 DB를 봐도 서로의 테이블이 보이지 않는다.

In [ ]:
spark.sql("SHOW NAMESPACES IN iceberg").show()

In [ ]:
# 네임스페이스별 테이블 — 위 결과에 맞춰 바꿔 쓴다.
# usgs_water는 공개 데이터(미국 지질조사국 수문 관측)라 반출 제약이 없다.
NAMESPACE = "usgs_water"

spark.sql(f"SHOW TABLES IN iceberg.{NAMESPACE}").show()

## 4. 조회 → pandas

EDA는 pandas로 넘겨서 한다. **`toPandas()`는 드라이버 메모리에 전량 적재**하므로
반드시 집계하거나 `limit`을 건 뒤에 호출한다.

In [ ]:
TABLE = f"iceberg.{NAMESPACE}.water_iv_raw"

df = spark.sql(f"SELECT * FROM {TABLE} LIMIT 100").toPandas()
df.head()

## 5. Iceberg 메타데이터

스냅샷·파일 목록은 메타데이터 테이블로 본다. small-files 누적을 확인할 때 쓴다
(유지보수 정책은 `docs/operations.md`).

In [ ]:
spark.sql(f"SELECT * FROM {TABLE}.snapshots ORDER BY committed_at DESC").show(
    5, truncate=False
)

## 6. (선택) pyiceberg 직접 접속

Spark를 거치지 않고 pyarrow로 바로 읽는 경로. **Dagster 적재 에셋과 같은 코드**를
쓰므로 적재 결과를 그대로 재현·검증할 수 있다.

접속 파라미터의 단일 출처는 `common.constants`이고, 대상 전환은 **`.env`로** 한다.
호스트에서 K8s 카탈로그를 보려면 `.env`에 아래가 채워져 있어야 한다
(키 목록은 `.env.example` 참고 — 값은 커밋하지 않는다):

```
ICEBERG_CATALOG_HOST=localhost
ICEBERG_CATALOG_PORT=15432
ICEBERG_CATALOG_DB=iceberg
ICEBERG_CATALOG_USER=iceberg
ICEBERG_CATALOG_PASSWORD=...      # k8s Secret catalog-pg-app 의 password
ICEBERG_S3_ENDPOINT=http://localhost:18333
ICEBERG_S3_ACCESS_KEY=...         # k8s Secret lakehouse-creds 의 s3-access-key
ICEBERG_S3_SECRET_KEY=...         # k8s Secret lakehouse-creds 의 s3-secret-key
```

시크릿이 **용도별로 나뉘어 있다** — PG 계정은 `catalog-pg-app`(CloudNativePG bootstrap 시크릿,
`basic-auth`라 키 이름이 `username`/`password`로 고정), S3 키는 `lakehouse-creds`다.
같은 비밀번호를 두 시크릿에 중복 보관하지 않는 것이 규약이다(`docs/conventions/k8s.md` §12).

🔴 **엔드포인트와 S3 키는 한 쌍이다.** 엔드포인트만 K8s로 바꾸고 키를 공용
`AWS_ACCESS_KEY_ID`(compose SeaweedFS용)로 두면 **네임스페이스·테이블 나열까지는
성공하고 `load_table`에서 `ACCESS_DENIED`** 로 죽는다(2026-08-19 실측).
부분 성공이라 원인을 오해하기 쉽다.
`ICEBERG_S3_*`를 비우면 공용 `AWS_*`로 폴백한다(compose 단독 구성용).

카탈로그 키가 미설정이면 기본값이 **compose**(`postgres:5432/iceberg_catalog`)를
가리켜 호스트에서는 이름 해석에 실패한다.

In [ ]:
# .env는 §1에서 이미 읽었다(load_dotenv). 여기서 다시 부르지 않는다.
from dagster_project.common.constants import (
    AWS_REGION,
    CATALOG_NAME,
    ICEBERG_CATALOG_URI,
    S3_ACCESS_KEY_ID,
    S3_ENDPOINT,
    S3_SECRET_ACCESS_KEY,
    WAREHOUSE,
)
from pyiceberg.catalog.sql import SqlCatalog

catalog = SqlCatalog(
    CATALOG_NAME,
    **{
        "uri": ICEBERG_CATALOG_URI,
        "warehouse": WAREHOUSE,
        "s3.endpoint": S3_ENDPOINT,
        # ICEBERG_S3_* 가 있으면 그 값, 없으면 공용 AWS_* 로 폴백된 값이다.
        "s3.access-key-id": S3_ACCESS_KEY_ID,
        "s3.secret-access-key": S3_SECRET_ACCESS_KEY,
        "s3.region": AWS_REGION,
        "s3.path-style-access": "true",
    },
)
print(catalog.list_namespaces())

## 7. 정리

Spark Connect는 **상주 컴퓨트**다. `--master k8s://` 이후로는 **executor 파드도 함께 상주**한다
(`spark.executor.instances`는 정적 할당이라 세션이 살아 있는 내내 산다).

```shell
kubectl scale deploy/spark-connect --replicas=0
kubectl get pods -l spark-role=executor       # 🔴 executor도 함께 사라져야 한다
```

🔴 **executor가 남으면 회수가 안 된 것이다.** driver 파드를 소유자로 걸어 두지 않으면
(`spark.kubernetes.driver.pod.name`) `--replicas=0`이 driver만 내리고 executor는 남는데,
에러도 알림도 없이 1 CPU를 계속 점유한다(`docs/conventions/k8s.md` §9-3).

In [ ]:
spark.stop()